### Проверка, что S3 и Spark работают

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("SCD2_Homework")
         .master("spark://spark-master:7077")
         .getOrCreate())

print("Spark version:", spark.version)

Spark version: 3.5.0


In [ ]:
data_test = [("test_user", 1), ("another_user", 2)]
df_test = spark.createDataFrame(data_test, ["name", "value"])

df_test.write.mode("overwrite").parquet("s3a://dwh/_test/test.parquet")

In [2]:
df_read = spark.read.parquet("s3a://dwh/_test/test.parquet")
df_read.show()

+------------+-----+
|        name|value|
+------------+-----+
|another_user|    2|
|   test_user|    1|
+------------+-----+



### Создание искусственных данных

In [4]:
! python generate_event_log.py --rows 5000 --output-dir ./event_log --seed 42

files=5/5 rows=5000/5000 rate=101,187 rows/s
Готово: 5000 строк в каталоге event_log


In [3]:
import pandas as pd

df_events = pd.read_csv("event_log/events_00001.csv")
print(df_events.head(5))

             event_time  user_id         user_fio  \
0  2026-09-01T00:00:03Z       29     Aiden Howard   
1  2026-09-01T00:00:04Z       36  Ashley Williams   
2  2026-09-01T00:00:05Z      152    Joseph Barnes   
3  2026-09-01T00:00:06Z       56       David Reed   
4  2026-09-01T00:00:07Z      144       Chloe Ward   

                           user_email   user_login  
0        aiden.howard.29@mail.example    ahoward29  
1  ashley.williams.36@webmail.example  awilliams36  
2       joseph.barnes.152@example.com   jbarnes152  
3       david.reed.56@webmail.example      dreed56  
4      chloe.ward.144@webmail.example     cward144  


### Загрузка данных в S3 была сделана через Rustfs Web-консоль. 
Выведем сырые данные из S3:

In [4]:
event_test_read = spark.read.option("header", "true").csv("s3a://dwh/source/event_log/*.csv")
event_test_read.show(5, truncate=False)

+--------------------+-------+-----------------+--------------------+-----------+
|          event_time|user_id|         user_fio|          user_email| user_login|
+--------------------+-------+-----------------+--------------------+-----------+
|2026-09-01T02:13:35Z|     44|  Caroline Butler|caroline.butler.4...|  cbutler44|
|2026-09-01T02:13:36Z|     18|   Dylan Thompson|dylan.thompson.18...|dthompson18|
|2026-09-01T02:13:38Z|     91|Christopher White|christopher.white...|   cwhite91|
|2026-09-01T02:13:39Z|    182|    Taylor Fisher|taylor.fisher.182...| tfisher182|
|2026-09-01T02:13:41Z|    122|   Cameron Watson|cameron.watson.12...| cwatson122|
+--------------------+-------+-----------------+--------------------+-----------+
only showing top 5 rows



### Реализация SCD2
под SCD2 подразумевается, что при обновлении данных в таблице будет создаваться новая запись в БД, а не просто обновляться предыдущие записи. Благодаря этому реализуется версионность и сохранение архивных данных. Также добавляются новые служебные столбцы:
1) "valid_from" - временная метка, начиная с которой эта запись актульна/была актуальна
2) "valid_to"   - временная метка, показывающая потерю актуальности этой записи. Если valid_to=NULL, то запись актуальна
3) "is_current" - дополнительная метка, которая показывает актуальность записи. По сути полностью повторяет смысл valid_to=NULL, но с ним нагляднее

In [5]:
source_path = "s3a://dwh/source/event_log/"
df_raw = (
    spark.read
    .option("header", "true")
    .option("mode",
            "FAILFAST")  # если в читаемом CSV найдётся некорректное поле, то вызовется ошибка, а не просто замена на NULL
    .csv(source_path)
)

print("Количество строк:", df_raw.count())
df_raw.printSchema()

Количество строк: 5000
root
 |-- event_time: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)



In [6]:
from pyspark.sql import functions as F

# преобразуем столбцы в нужные типы, т.к. после прочтения они все оказались string
df_events = df_raw.select(
    F.to_timestamp(
        F.col("event_time"),
        "yyyy-MM-dd'T'HH:mm:ss'Z'"
    ).alias("event_time"),
    F.col("user_id").cast("long").alias("user_id"),  # используем long (64 бита), чтобы полностью уместить uint32
    F.col("user_fio").cast("string").alias("user_fio"),
    F.col("user_email").cast("string").alias("user_email"),
    F.col("user_login").cast("string").alias("user_login")
)

df_events.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)



In [8]:
# проверяем полученные данные, чтобы понять как можно с ними работать
null_rows = df_events.filter(
    F.col("event_time").isNull() |
    F.col("user_id").isNull() |
    F.col("user_fio").isNull() |
    F.col("user_email").isNull() |
    F.col("user_login").isNull()
).count()
print(f"Кол-во строк c NULL: {null_rows}")

total = df_events.count()
distinct = df_events.distinct().count()
print(f"Кол-во полных дубликатов строк: {total - distinct}")

Кол-во строк дубликатов: 0
Кол-во строк c NULL: 0


In [10]:
from pyspark.sql.window import Window

# получаем отдельные последовательности событий для каждого user_id, отсортированных по event_time
window_user = (
    Window
    .partitionBy("user_id")
    .orderBy("event_time")
)

# объединяем столбцы "user_fio", "user_email" и "user_login" в общие структуры, чтобы было легче отслеживать изменение параметров
df_with_profile = (
    df_events
    .withColumn(
        "_profile",
        F.struct(
            "user_fio",
            "user_email",
            "user_login"
        )
    )
    .withColumn(
        "_previous_profile",
        F.lag("_profile").over(window_user) # lag() - получает значение из предыдущих строк датафрейма в рамках заданного окна
    )
)

# оставляем только те строки, которые изначально равны NULL (первая запись для этого user_id)
# или те, у которых поменялся какой-то параметр из структуры
df_changed = df_with_profile.filter(
    F.col("_previous_profile").isNull() |
    (
        ~F.col("_profile").eqNullSafe(
            F.col("_previous_profile")
        )
    )
)

# приводим df_changed в вид нормальной таблицы и
# переименовываем столбец "event_time" в "valid_from"
df_versions = df_changed.select(
    "user_id",
    "user_fio",
    "user_email",
    "user_login",
    F.col("event_time").alias("valid_from")
)

# создаём новое окно, т.к. переименовали столбец
window_versions = (
    Window
    .partitionBy("user_id")
    .orderBy("valid_from")
)

# к полученной таблице добавляем столбцы "valid_to" и "is_current"
df_scd2 = (
    df_versions
    .withColumn("valid_to", F.lead("valid_from").over(window_versions))
    .withColumn("is_current", F.col("valid_to").isNull())
)

In [14]:
# Выводим полученнуб таблицу в SCD2
df_scd2.printSchema()
df_scd2.orderBy(
    "user_id",
    "valid_from"
).show(30, truncate=False)

root
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = false)

+-------+------------------+----------------------------------+------------+-------------------+-------------------+----------+
|user_id|user_fio          |user_email                        |user_login  |valid_from         |valid_to           |is_current|
+-------+------------------+----------------------------------+------------+-------------------+-------------------+----------+
|1      |Jennifer Henderson|jennifer.henderson.1@post.example |jhenderson1 |2026-09-01 00:01:53|NULL               |true      |
|2      |Ashley Murphy     |ashley.murphy.2@mail.example      |amurphy2    |2026-09-01 00:04:14|2026-09-01 01:21:20|false     |
|2      |Taylor Scott      |ashley.murphy.2@mail.example     

### Сохранение DataFrame в SCD2 в хранилище S3
в выводе ячейки выше видим, что 2 юзера имеют по 2 записи и у них корректно прописаны столбцы "valid_from", "valid_to" и "is_current"

In [16]:
target_path = "s3a://dwh/target/users_scd2/"

# mode("overwrite") как раз обеспечивает идемпотентность. То есть при многократном выполнении одного и того же действия
# результат будет таким же как после первого выполнения этого действия.
# однако этот режим работает только в этом случае, когда мы за раз обрабатываем и записываем все данные.
# если бы мы работали в инкрементальном режиме, то нужно было бы использовать mode("append")
df_scd2.write.mode("overwrite").parquet(target_path)

In [17]:
df_scd2_result = spark.read.parquet(target_path)
df_scd2_result.printSchema()
df_scd2_result.orderBy(
    "user_id",
    "valid_from"
).show(30, truncate=False)

root
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)

+-------+------------------+----------------------------------+------------+-------------------+-------------------+----------+
|user_id|user_fio          |user_email                        |user_login  |valid_from         |valid_to           |is_current|
+-------+------------------+----------------------------------+------------+-------------------+-------------------+----------+
|1      |Jennifer Henderson|jennifer.henderson.1@post.example |jhenderson1 |2026-09-01 00:01:53|NULL               |true      |
|2      |Ashley Murphy     |ashley.murphy.2@mail.example      |amurphy2    |2026-09-01 00:04:14|2026-09-01 01:21:20|false     |
|2      |Taylor Scott      |ashley.murphy.2@mail.example      

Получили те же самые данные, значит всё сохранилось в S3 корректно

### Проверка на выполнение SCD2

In [18]:
# проверка, что у каждого юзера должна быть ровно одна запись с current_count=True
check_current = (
    df_scd2_result
        .groupBy("user_id")
        .agg(
            F.sum(
                F.col("is_current").cast("int")
            ).alias("current_count")
        )
        .filter(F.col("current_count") != 1)
)
print(f"Кол-во множественных актуальных записей: {check_current.count()}")

# проверка, что для каждой записи с current_count=True поле обязательно долно быть valid_to=NULL
check_current_flag = df_scd2_result.filter(
    F.col("is_current") != F.col("valid_to").isNull()
)
print(f"Кол-во несовпадений current_count и valid_to в актуальных записях: {check_current_flag.count()}")

# проверка, что "valid_from" не может иметь более ранюю метку чем "valid_to"
check_intervals = df_scd2_result.filter(
    F.col("valid_to").isNotNull() &
    (F.col("valid_to") <= F.col("valid_from"))
)
print(f"Кол-во некорректных временных интервалов: {check_intervals.count()}")

Кол-во множественных актуальных записей: 0
Кол-во несовпадений current_count и valid_to в актуальных записях: 0
Кол-во некорректных временных интервалов: 0


### Запрос на подсчёт кол-ва строк в полученной SCD2 таблице
<a name="scd2-count"></a>

In [21]:
# указываем ключ паркета в S3
target_path = "s3a://dwh/target/users_scd2/"
# читаем весь паркет
scd2_count_result = spark.read.parquet(target_path)
# подсчитываем число строк
count_result = scd2_count_result.count()
print(f"Число строк в SCD2 таблице: {count_result}")

Число строк в SCD2 таблице: 285


### Запрос который находит активный user_login пользователя X на дату и время Y
<a name="scd2-login"></a>

In [24]:
def find_login_by_id_date(df, user_id, timestamp):
    # преобразуем переданную дату в timestamp
    timestamp = F.to_timestamp(F.lit(timestamp))

    # выбираем актуальную запись пользователя
    return (
        df.filter(
            (F.col("user_id") == F.lit(user_id)) &
            (F.col("valid_from") <= timestamp) &
            (
                F.col("valid_to").isNull() |
                (timestamp < F.col("valid_to"))
            )
        )
        .select("user_id", "user_login")
    )

In [27]:
result = find_login_by_id_date(
    df_scd2_result,
    user_id=2,
    timestamp="2026-09-01T00:30:00Z"
)
result.show()

# пустый вывод, т.к. неправильная дата
result = find_login_by_id_date(
    df_scd2_result,
    user_id=2,
    timestamp="1926-09-01T00:30:00Z"
)
result.show()

# пустый вывод, т.к. неправильный id
result = find_login_by_id_date(
    df_scd2_result,
    user_id=2000,
    timestamp="2026-09-01T00:30:00Z"
)
result.show()

+-------+----------+
|user_id|user_login|
+-------+----------+
|      2|  amurphy2|
+-------+----------+

+-------+----------+
|user_id|user_login|
+-------+----------+
+-------+----------+

+-------+----------+
|user_id|user_login|
+-------+----------+
+-------+----------+

